In [35]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math


## Transformer Encoder trong PhoBERT
Ghép `TransformerEmbeddings` (01) và `EncoderBlock` (03 , gồm `MultiHeadAttention` (02) + Feed Forward + Residual/LayerNorm) rồi xếp chồng N lớp để tạo mô hình PhoBERT rút gọn, đầu ra là Contextual Embeddings

### Tái sử dụng từ các notebook trước (01,02 và 03)

In [36]:
#Thực hiện việc chuyển đổi từ ngữ thành vector (Token Embedding) và cộng thêm thông tin vị trí (Positional Embedding)
class TransformerEmbeddings(nn.Module):
    def __init__(self, vocab_size, d_model=512, max_len=5000):
        super().__init__()
        #Token Embedding: Mã hóa ý nghĩa
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        
        #Positional Embedding: Mã hóa vị trí
        self.positional_encoding = nn.Parameter(torch.zeros(1, max_len, d_model))
        
    def forward(self, x):
        # x: (batch_size, seq_len)
        tokens = self.token_embedding(x)
        #Cộng Token Embedding và Positional Embedding 
        return tokens + self.positional_encoding[:, :tokens.size(1), :]


In [37]:
#Ví dụ linh hồn của Transformer: Multi-Head Attention với cơ chế Query, Key, Value
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model=512, n_heads=8):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model = d_model
        self.d_k = d_model // n_heads # 64 chiều mỗi đầu
        self.n_heads = n_heads
        
        #Các ma trận trọng số WQ, WK, WV
        self.w_q = nn.Linear(d_model, d_model)
        self.w_k = nn.Linear(d_model, d_model)
        self.w_v = nn.Linear(d_model, d_model)
        self.w_o = nn.Linear(d_model, d_model)

    def forward(self, q, k, v, mask=None):
        batch_size = q.size(0)
        
        #Tạo Q, K, V và chia thành n_heads
        Q = self.w_q(q).view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)
        K = self.w_k(k).view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)
        V = self.w_v(v).view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)

        #Tính Scores bằng tích vô hướng và Scaling
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)

        #Softmax để lấy trọng số chú ý
        attn = F.softmax(scores, dim=-1)
        
        #Nhân với Value và gộp các đầu lại
        context = torch.matmul(attn, V).transpose(1, 2).contiguous().view(batch_size, -1, self.d_model)
        return self.w_o(context)


In [38]:
#Mạng Feed Forward và Residual Connection để tín hiệu không bị hao hụt
#Định nghĩa mạng Feed Forward (FFN)
#Gồm 2 phép biến đổi tuyến tính và ReLU ở giữa
class FeedForward(nn.Module):
    def __init__(self, d_model=512, d_ff=2048):
        super().__init__()
        # Mở rộng từ 512 lên 2048 chiều
        self.linear1 = nn.Linear(d_model, d_ff)
        self.relu = nn.ReLU()
        # Ánh xạ ngược về lại 512 chiều
        self.linear2 = nn.Linear(d_ff, d_model)

    def forward(self, x):
        return self.linear2(self.relu(self.linear1(x)))


In [39]:
#Khối Encoder hoàn chỉnh (Encoder Block)
class EncoderBlock(nn.Module):
    def __init__(self, d_model=512, n_heads=8, d_ff=2048):
        super().__init__()
        #Dùng MultiHeadAttention
        self.self_attention = MultiHeadAttention(d_model=d_model, n_heads=n_heads)
        self.norm1 = nn.LayerNorm(d_model) # Chuẩn hóa lớp 1
        
        self.ffn = FeedForward(d_model, d_ff)
        self.norm2 = nn.LayerNorm(d_model) # Chuẩn hóa lớp 2

    def forward(self, x):
        #Self-Attention + Residual Connection (Thang máy)
        # x + Sublayer(x)
        attention_res = self.norm1(x + self.self_attention(x, x, x))
        
        #Feed Forward + Residual Connection
        #Giúp tinh chỉnh lại biểu diễn của từng token
        final_out = self.norm2(attention_res + self.ffn(attention_res))
        
        return final_out


### Xếp chồng N `EncoderBlock` thành một mô hình hoàn chỉnh

In [40]:
class PhoBERTEncoderToyModel(nn.Module):
    def __init__(self, vocab_size=1000, d_model=512, n_heads=8, d_ff=2048, num_layers=2):
        super().__init__()
        #Dùng lại TransformerEmbeddings
        self.embeddings = TransformerEmbeddings(vocab_size=vocab_size, d_model=d_model)

        # Xếp chồng N lớp Encoder tự xây dựng
        self.layers = nn.ModuleList([
            EncoderBlock(d_model=d_model, n_heads=n_heads, d_ff=d_ff) for _ in range(num_layers)
        ])

    def forward(self, x):
        # x: (batch_size, seq_len)
        x_emb = self.embeddings(x)
        #Lần lượt đi qua từng Encoder Layer
        for layer in self.layers:
            x_emb = layer(x_emb)
        #Trả về Contextual Embeddings cho từng token
        return x_emb


In [41]:
#Cấu hình Demo
sentence = "You are a beautiful girl"
tokens = sentence.split() # 5 từ
vocab = {"You": 101, "are": 102, "a": 103, "beautiful": 104, "girl": 105}
input_ids = torch.tensor([[vocab[w] for w in tokens]]) # Shape: (1, 5)

# Khai báo mô hình PhoBERT
phobert_model = PhoBERTEncoderToyModel(vocab_size=1000, d_model=512, num_layers=2)
contextual_vectors = phobert_model(input_ids)

print(f"Câu đầu vào: '{sentence}'")
print(f"Kích thước Contextual Vector đầu ra: {contextual_vectors.shape}")


Câu đầu vào: 'You are a beautiful girl'
Kích thước Contextual Vector đầu ra: torch.Size([1, 5, 512])
